# MarineMamba: Full Experiment Pipeline

**Self-contained notebook** — fetches BOLD data, processes it, runs all 6 models.

| Model | Method | GPU Needed |
|-------|--------|------------|
| A | BLAST baseline | CPU |
| B | k-NN + 6-mer | CPU |
| C | BarcodeMamba transfer (insect→fish) | T4/A100 |
| D | BarcodeMamba from scratch | T4/A100 |
| E | BarcodeMamba domain-adapted | T4/A100 |
| F | Evo 2 (7B) embeddings | A100 only |

**Runtime**: 2025.10 + A100 (Colab Pro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless13/marinemamba/blob/main/notebooks/gpu_runner.ipynb)

---
## Cell 1: Environment Setup

**IMPORTANT**: Use Runtime version **2025.10** (not Latest). Go to Runtime → Change runtime type → A100 + 2025.10.

This cell upgrades setuptools (needed for vtx), then restarts the runtime. **After restart, skip to Cell 2.**

In [ ]:
# Step 1: Upgrade setuptools (vtx needs license-files support)
!pip install --upgrade setuptools

# Restart runtime — required before installing CUDA packages
import os
os.kill(os.getpid(), 9)

---
## Cell 2: Install Packages

**Run this after runtime restart.** Install order matters — flash-attn and vtx before mamba-ssm.

In [ ]:
%%time
# Install order is critical — do not change
!pip install flash-attn --no-build-isolation 2>&1 | tail -3
!pip install vtx --no-build-isolation 2>&1 | tail -3
!pip install evo2 --no-build-isolation 2>&1 | tail -3
!pip install mamba-ssm --no-build-isolation 2>&1 | tail -3
!pip install causal-conv1d --no-build-isolation 2>&1 | tail -3
!pip install pytorch_lightning einops timm torchtext transformers huggingface_hub tqdm scikit-learn 2>&1 | tail -3

# Clone repo
!git clone https://github.com/kluless13/marinemamba.git 2>/dev/null; true
%cd /content/marinemamba

# Verify
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"PyTorch: {torch.__version__}")

from mamba_ssm.modules.mamba2 import Mamba2
from causal_conv1d import causal_conv1d_fn
from evo2 import Evo2
print("\nmamba-ssm: OK")
print("causal-conv1d: OK")
print("evo2: OK")
print("\nAll packages installed successfully!")

---
## Cell 3: Fetch & Process BOLD Data

Downloads 420K+ Teleostei (bony fish) COI barcodes from the [BOLD v5 API](https://portal.boldsystems.org/api), then processes into train/test/unseen splits.

In [ ]:
%%time
import requests
import csv
import sys
import json
import os
from pathlib import Path
from collections import Counter

csv.field_size_limit(sys.maxsize)

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

BOLD_TSV = RAW_DIR / "bold_teleostei.tsv"
MERGED_CSV = RAW_DIR / "merged_barcodes.csv"

# ── Step 1: Download from BOLD v5 API ─────────────────────────────────────
if not BOLD_TSV.exists():
    print("Fetching Teleostei from BOLD v5 API...")
    BASE = "https://portal.boldsystems.org/api"
    
    r = requests.get(f"{BASE}/query", params={"query": "tax:class:Teleostei", "extent": "full"}, timeout=60)
    r.raise_for_status()
    query_id = r.json()["query_id"]
    print(f"  Query ID: {query_id}")
    
    r = requests.get(f"{BASE}/documents/{query_id}/download", params={"format": "tsv"}, timeout=600, stream=True)
    r.raise_for_status()
    with open(BOLD_TSV, "wb") as f:
        total = 0
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            total += len(chunk)
    print(f"  Downloaded: {total / 1e6:.1f} MB")
else:
    print(f"BOLD data already exists: {BOLD_TSV}")

# Count raw records
with open(BOLD_TSV) as f:
    raw_count = sum(1 for _ in f) - 1
print(f"  Raw records: {raw_count:,}")

# ── Step 2: Convert to merged_barcodes.csv ────────────────────────────────
print("\nConverting BOLD TSV → merged_barcodes.csv...")
kept = 0
skipped = Counter()

with open(BOLD_TSV, "r") as infile, open(MERGED_CSV, "w", newline="") as outfile:
    reader = csv.DictReader(infile, delimiter="\t")
    writer = csv.DictWriter(outfile, fieldnames=["processid", "nucleotides", "species_name", "genus_name", "family_name", "order_name"])
    writer.writeheader()
    
    for row in reader:
        seq = row.get("nuc", "").strip()
        species = row.get("species", "").strip()
        marker = row.get("marker_code", "").strip()
        
        if "COI" not in marker.upper():
            skipped["no_COI"] += 1
            continue
        if not seq or len(seq) < 100:
            skipped["short_seq"] += 1
            continue
        if not species or " " not in species:
            skipped["no_species"] += 1
            continue
        
        genus = row.get("genus", "").strip() or species.split()[0]
        writer.writerow({
            "processid": row.get("processid", "").strip(),
            "nucleotides": seq.upper().replace("-", "").replace(".", ""),
            "species_name": species,
            "genus_name": genus,
            "family_name": row.get("family", "").strip(),
            "order_name": row.get("order", "").strip(),
        })
        kept += 1

print(f"  Kept: {kept:,}")
for reason, count in skipped.most_common():
    print(f"  Skipped ({reason}): {count:,}")

# ── Step 3: Run train/test/unseen splits ──────────────────────────────────
print("\nRunning train/test/unseen splits...")
!python scripts/02_clean_and_split.py

---
## Cell 4: Models A & B — Baselines (BLAST + k-NN)

CPU-only. BLAST may not be installed on Colab — if so, we skip it and only run k-NN + Random Forest.

In [ ]:
%%time
# Install BLAST (optional, may fail on Colab)
!apt-get install -qq ncbi-blast+ 2>/dev/null || echo "BLAST not available — will skip"

!python scripts/03_baselines.py

---
## Cell 5: Model C — BarcodeMamba Transfer (insect → fish)

Loads the insect-pretrained BarcodeMamba checkpoint from HuggingFace, fine-tunes on fish COI barcodes.

In [ ]:
%%time
!python scripts/04_barcodemamba_models.py --mode transfer --data-dir data/processed --output-dir results

---
## Cell 6: Model D — BarcodeMamba from Scratch

Pretrains on fish COI barcodes (NTP objective), then fine-tunes. Longest run (~12-20h on T4, ~4-6h on A100).

In [ ]:
%%time
!python scripts/04_barcodemamba_models.py --mode scratch --data-dir data/processed --output-dir results

---
## Cell 7: Model E — BarcodeMamba Domain Adaptation

Takes the insect checkpoint, continues pretraining on fish COI, then fine-tunes.

In [ ]:
%%time
!python scripts/04_barcodemamba_models.py --mode adapt --data-dir data/processed --output-dir results

---
## Cell 8: Model F — Evo 2 (7B) Embeddings

Loads the 13.8GB Evo 2 7B model, extracts embeddings, trains a linear classifier.

**Requires A100.** Clear GPU memory before running:

In [ ]:
%%time
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

!python scripts/05_evo2_embeddings.py --data-dir data/processed --output-dir results

---
## Cell 9: Results Summary

In [ ]:
import json, glob

# Load dataset stats
with open("data/processed/dataset_stats.json") as f:
    stats = json.load(f)

print("=" * 70)
print("MARINEMAMBA EXPERIMENT RESULTS")
print("=" * 70)
print(f"\nDataset: BOLD Teleostei COI Barcodes")
print(f"  Total sequences: {stats['total_sequences']:,}")
print(f"  Species:         {stats['total_species']:,}")
print(f"  Genera:          {stats['total_genera']:,}")
print(f"  Train:           {stats['train_size']:,}")
print(f"  Test:            {stats['test_size']:,}")
print(f"  Unseen genera:   {stats['unseen_size']:,} ({stats['unseen_genera']} genera)")
print(f"  N classes:       {stats['n_classes']:,}")

print(f"\n{'Model':<35} {'Species Acc':>12} {'Unseen Genus':>14}")
print("-" * 65)

for f_path in sorted(glob.glob("results/*.json")):
    name = f_path.split("/")[-1].replace(".json", "")
    with open(f_path) as f:
        r = json.load(f)
    
    if name == "baselines":
        blast_acc = r.get("blast", {}).get("accuracy", "N/A")
        knn_sp = r.get("knn", {}).get("species_accuracy_test", "N/A")
        knn_g = r.get("knn", {}).get("genus_accuracy_unseen", "N/A")
        rf_acc = r.get("random_forest", {}).get("species_accuracy_test", "N/A")
        
        if blast_acc != "N/A":
            print(f"{'A: BLAST':<35} {blast_acc:>12.4f} {'0.0000':>14}")
        print(f"{'B: k-NN (6-mer, cosine)':<35} {knn_sp:>12.4f} {knn_g:>14.4f}")
        print(f"{'B: Random Forest (6-mer)':<35} {rf_acc:>12.4f} {'—':>14}")
    else:
        sp_acc = r.get("linear_probe_accuracy", r.get("species_logistic_regression", {}).get("accuracy", "?"))
        g_acc = r.get("genus_accuracy_unseen", r.get("genus_knn_unseen", {}).get("accuracy", "?"))
        label = name.replace("_results", "").replace("_", " ").title()
        if isinstance(sp_acc, float):
            sp_str = f"{sp_acc:>12.4f}"
        else:
            sp_str = f"{sp_acc:>12}"
        if isinstance(g_acc, float):
            g_str = f"{g_acc:>14.4f}"
        else:
            g_str = f"{g_acc:>14}"
        print(f"{label:<35} {sp_str} {g_str}")

print("=" * 65)

---
## Cell 10: Save Results

Push results to GitHub and optionally save to Google Drive.

In [ ]:
# Push results to GitHub
!git add -f results/*.json data/processed/dataset_stats.json
!git commit -m "results: BOLD 320K dataset, all 6 models"
!git push

# Optional: save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/marinemamba/
    !cp -r results/ /content/drive/MyDrive/marinemamba/results/
    !cp -r checkpoints/ /content/drive/MyDrive/marinemamba/checkpoints/
    !cp data/processed/dataset_stats.json /content/drive/MyDrive/marinemamba/
    print("Saved to Google Drive")
except Exception as e:
    print(f"Drive save skipped: {e}")